# Arabic Fake News Detection — ARABICFAKETWEETS / MARBERTv2

**Pipeline stages:**
1. Environment setup
2. Dataset loading (ARABICFAKETWEETS — Mendeley CSV format)
3. Tweet preprocessing (lighter than formal news — keep dialectal features)
4. Label encoding & class-weight computation
5. 80/10/10 stratified splits
6. PyTorch Dataset & tokenisation
7. WeightedTrainer + metrics (same as Notebook 1)
8. MARBERTv2 training (3 epochs)
9. Evaluation & confusion matrix
10. LIME explainability (batched)
11. Inference detector (drop-in compatible with Notebook 1 detector class)
12. Inference timing benchmark
13. Export to Hugging Face Hub
14. Disk-reload sanity checks

> **Runtime notes**
> - Developed and run on Kaggle (GPU T4/P100). All Kaggle-specific paths (`/kaggle/input`, `/kaggle/working`) and the `kaggle_secrets` import are guarded with fallbacks, but you should adjust `find_file()`'s search roots and the dataset upload step if running elsewhere (e.g. locally or on Colab).
> - Hugging Face Hub export (Cell 15) requires an `HF_TOKEN`. On Kaggle this is read from Kaggle Secrets; outside Kaggle, call `login(token="hf_...")` manually.
> - Install dependencies with `pip install pyarabic lime transformers torch scikit-learn pandas numpy matplotlib seaborn tqdm` if not already present.


## Cell 1 — Install Dependencies

In [ ]:
!pip install -q pyarabic
!pip install -q lime

print('Dependencies ready.')

## Cell 2 — Imports & Global Config

In [ ]:
import os, re, json, time, warnings, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

import torch
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    pipeline as hf_pipeline,
)
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix
)
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

import pyarabic.araby as araby

# ── Reproducibility ────────────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# ── Device ─────────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU : {torch.cuda.get_device_name(0)}')
    print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# ── Hyperparameters ────────────────────────────────────────────────────────────

MAX_LEN = 64 
BATCH_SIZE = 32 
EPOCHS = 3 
LR = 2e-5 
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1

# ── Quick-test mode ────────────────────────────────────────────────────────────
MAX_SAMPLES = None 
# ── Model identifier ───────────────────────────────────────────────────────────
MARBERT_ID = 'UBC-NLP/MARBERTv2'

# ── Output directory ───────────────────────────────────────────────────────────
OUT_MARBERT = '/kaggle/working/model_marbertv2'
os.makedirs(OUT_MARBERT, exist_ok=True)

# ── Label maps ─────────────────────────────────────────────────────────────────
LABEL2ID = {'credible': 1, 'not_credible': 0}
ID2LABEL = {0: 'not_credible', 1: 'credible'}

print('\nConfig ready.')
print(f'MAX_LEN={MAX_LEN}, BATCH_SIZE={BATCH_SIZE}, EPOCHS={EPOCHS}, LR={LR}')

## Cell 3 — Load ARABICFAKETWEETS Dataset

 Mendeley direct link: https://data.mendeley.com/datasets/9sht4t6cpf/2


In [ ]:
def find_file(name_fragments, search_roots=('/kaggle/input', '/kaggle/working')):
    """Search for a file whose path contains any of the given name fragments."""
    if isinstance(name_fragments, str):
        name_fragments = [name_fragments]
    for root in search_roots:
        for dirpath, _, files in os.walk(root):
            for f in files:
                fp = os.path.join(dirpath, f).lower()
                if any(frag.lower() in fp for frag in name_fragments):
                    return os.path.join(dirpath, f)
    return None


def load_tweets_csv(csv_path):
    """
    Load ARABICFAKETWEETS from CSV.
    Handles multiple column naming conventions seen across Mendeley/Kaggle mirrors.
    Returns DataFrame with columns: text, label_str
    """
    df = pd.read_csv(csv_path, low_memory=False)
    print(f'Raw CSV columns : {list(df.columns)}')
    print(f'Raw CSV shape : {df.shape}')

    cols_lower = {c.lower().strip(): c for c in df.columns}

    # ── Find text column ──
    text_col = None
    for cand in ['text', 'tweet', 'content', 'body', 'news_text', 'sentence']:
        if cand in cols_lower:
            text_col = cols_lower[cand]
            break
    assert text_col, f'Cannot find text column. Available: {list(df.columns)}'

    # ── Find label column ──
    label_col = None
    for cand in ['label', 'class', 'target', 'credibility', 'category', 'fake', 'type']:
        if cand in cols_lower:
            label_col = cols_lower[cand]
            break
    assert label_col, f'Cannot find label column. Available: {list(df.columns)}'

    print(f'Using: text={text_col!r}, label={label_col!r}')

    df['_text'] = df[text_col].fillna('').astype(str).str.strip()
    df['_label_raw'] = df[label_col].astype(str).str.strip().str.lower()

    print(f'Raw label values: {df["_label_raw"].value_counts().to_dict()}')
    return df[['_text', '_label_raw']].rename(columns={'_text': 'text', '_label_raw': 'label_raw'})


# ── Attempt to load dataset ───────────────────────────────────────────────────
print('Searching for ARABICFAKETWEETS dataset...')

# Priority search order: most specific first
tweet_csv = find_file([
    'arabicfaketweets', 'arabic_fake_tweets', 'arabic-fake-tweets',
    'pocket_sheet', 'pocketsheet', 'arabic_tweets'
])

if tweet_csv:
    print(f'Found: {tweet_csv}')
    df_tweets_raw = load_tweets_csv(tweet_csv)
else:
    # ── Fallback: check if there are two separate CSV files (fake + real) ──────
    # Some mirrors split into fake_tweets.csv + real_tweets.csv
    fake_csv = find_file(['fake_tweet', 'fake-tweet'])
    real_csv = find_file(['real_tweet', 'real-tweet', 'no_rumor', 'norumor'])

    if fake_csv and real_csv:
        print(f'Found split CSVs: fake={fake_csv}, real={real_csv}')
        df_fake = pd.read_csv(fake_csv, low_memory=False)
        df_real = pd.read_csv(real_csv, low_memory=False)

        # Identify text column in each
        def get_text_col(df):
            for c in df.columns:
                if any(k in c.lower() for k in ['text', 'tweet', 'content', 'body']):
                    return c
            return df.columns[0]

        fc = get_text_col(df_fake)
        rc = get_text_col(df_real)
        df_fake = pd.DataFrame({'text': df_fake[fc].fillna('').astype(str), 'label_raw': 'not_credible'})
        df_real = pd.DataFrame({'text': df_real[rc].fillna('').astype(str), 'label_raw': 'credible'})
        df_tweets_raw = pd.concat([df_fake, df_real], ignore_index=True)
        print(f'Merged: {len(df_fake):,} fake + {len(df_real):,} real = {len(df_tweets_raw):,} total')
    else:
        # ── Demo fallback: synthetic tweet-style data ─────────────────────────
        print(f'ARABICFAKETWEETS not found. Using synthetic demo data (128 samples).')
        print(f'Add the dataset via Kaggle > Add Data for real training.')
        print(f'Mendeley: https://data.mendeley.com/datasets/9sht4t6cpf/2')
        print(f'Kaggle search: arabicfaketweets')
        demo = [
            # Fake tweets (informal, sensationalist, dialectal Arabic)
            ('بكره الحكومة حتوزع فلوس لكل المواطنين مش هتصدق الخبر ده', 'not_credible'),
            ('جرب العلاج ده وهتتشفى من السكر في اسبوع مضمون 100٪', 'not_credible'),
            ('صور جديدة تكشف المؤامرة الكبيرة ضد بلادنا والحكام بيخبوا الحقيقة', 'not_credible'),
            ('خبر عاجل: زلزال مدمر يضرب المنطقة وأكثر من 500 قتيل لحد دلوقتي', 'not_credible'),
            ('واحدة من الممثلات الشهيرة اعترفت بعمليات التجميل اللي عملتها كلها', 'not_credible'),
            ('تسريب خطير يكشف سرقة مليارات من خزينة الدولة بشهادة مصادر موثوقة', 'not_credible'),
            ('هذا الدواء يسبب الموت وشركات الدواء تخفيه عن الناس منذ سنوات طويلة', 'not_credible'),
            ('صور الأقمار الصناعية تكشف قاعدة عسكرية سرية في وسط الصحراء', 'not_credible'),
            # Real tweets (factual, formal or semi-formal)
            ('أعلنت وزارة الصحة عن تطعيم الأطفال ضد الإنفلونزا مجانا في المراكز الصحية', 'credible'),
            ('أطلقت الحكومة منصة رقمية جديدة لتسجيل الطلاب في الجامعات لهذا العام', 'credible'),
            ('قرر البنك المركزي تثبيت سعر الفائدة عند مستواه الحالي خلال الاجتماع الأخير', 'credible'),
            ('نظمت البلدية حملة لتنظيف الشواطئ بمشاركة المتطوعين من مختلف الأحياء', 'credible'),
            ('وقعت شركة اتصالات اتفاقية مع عدة دول لتوفير خدمة الإنترنت السريع بأسعار معقولة', 'credible'),
            ('أعلن المدير العام للشركة عن توظيف 500 شخص جديد خلال الربع القادم', 'credible'),
            ('صرح الناطق الرسمي بأن الاجتماع سيعقد الأسبوع القادم لمناقشة الميزانية', 'credible'),
            ('أشادت منظمة الصحة العالمية بجهود الدولة في مكافحة الأوبئة خلال السنوات الأخيرة', 'credible'),
        ] * 100
        df_tweets_raw = pd.DataFrame(demo, columns=['text', 'label_raw'])

print(f'\nRaw dataset: {len(df_tweets_raw):,} rows')
print('label_raw distribution:')
print(df_tweets_raw['label_raw'].value_counts())

## Cell 4 — Tweet Preprocessing

**Why lighter preprocessing than (AFND):**
- AFND = formal MSA news, so heavy normalization is safe (no dialectal features to preserve)
- ARABICFAKETWEETS = informal tweets, and **MARBERTv2 was pretrained on dialectal Arabic**, so we preserve dialectal forms, colloquialisms, and informal spelling
- We still remove noise (URLs, mentions, emojis) but do NOT aggressively normalize all Alef/Teh variants — the model has learned these dialectal forms

**Steps:**
1. Remove URLs, HTML, mentions (@), hashtag symbols (#)
2. Remove emojis
3. Strip diacritics with `araby.strip_tashkeel` (safe for all Arabic)
4. Strip tatweel (elongation) with `araby.strip_tatweel`
5. Normalize Alef variants to bare Alef (very common noise even in tweets)
6. Keep non-Arabic characters longer (numbers, some punctuation useful for tweets)
7. Collapse whitespace

**NOT done (unlike Notebook 1):**
- No Teh Marbuta normalization (dialectal distinction)
- No aggressive removal of all non-Arabic (some Latin chars appear in Arabic tweets legitimately)

In [ ]:
_EMOJI_PATTERN = re.compile(
    '[\U00010000-\U0010FFFF]',
    flags=re.UNICODE
)

# Additional Arabic-specific patterns
_REPEATED_CHARS = re.compile(r'(.)\1{2,}') # 3+ repeated chars 1

def clean_arabic_tweet(text: str) -> str:
    """
    Tweet-specific Arabic cleaning pipeline.
    LIGHTER than formal news cleaning — preserves dialectal features
    that MARBERTv2 was pretrained to understand.
    """
    if not isinstance(text, str) or not text.strip():
        return ''
    text = re.sub(r'http\S+|www\.\S+', ' ', text) # 1. URLs
    text = re.sub(r'<[^>]+>', ' ', text) # 2. HTML
    text = re.sub(r'@\S+', ' ', text) # 3. mentions
    text = text.replace('#', ' ') # 4. hashtag symbol (keep word)
    text = _EMOJI_PATTERN.sub(' ', text) # 5. emojis
    text = araby.strip_tashkeel(text) # 6. diacritics (safe for all Arabic)
    text = araby.strip_tatweel(text) # 7. elongation (حبيبيييي حبيبي)
    text = araby.normalize_alef(text) # 8. Alef variants (أإآ ا)
    # NOTE: NOT normalizing Teh Marbuta (dialectal feature MARBERTv2 understands)
    text = _REPEATED_CHARS.sub(r'\1\1', text) # 9. limit repeats (اهاهاها اهاه)
    text = re.sub(r'[^\u0600-\u06FF0-9a-zA-Z\s]', ' ', text) # 10. keep Arabic+digits+basic Latin
    text = re.sub(r'\s+', ' ', text).strip() # 11. whitespace
    return text


# ── Smoke test ─────────────────────────────────────────────────────────────────
samples = [
    'بكرهههههه الحكومة حتوزع فلوس!! http://fake.com #عاجل @الأخبار ',
    'العلاج ده مضمون 100٪ جربته بنفسي وتشفيت من السكر والضغط في اسبوع',
    'أعلنت وزارة الصحة عن بدء حملة التطعيمات في جميع المراكز الصحية',
]
print('Preprocessing smoke test (tweet cleaner):')
for s in samples:
    cleaned = clean_arabic_tweet(s)
    print(f'IN : {s}')
    print(f'OUT: {cleaned}')
    print()
print(f'Tweet preprocessing ready.')

In [ ]:
# ── Apply preprocessing ───────────────────────────────────────────────────────
print('Preprocessing tweets...')
tqdm.pandas(desc='Tweets')
df_tweets = df_tweets_raw.copy()
df_tweets['text'] = df_tweets['text'].progress_apply(clean_arabic_tweet)

# Remove empty/too-short rows (tweets < 5 chars after cleaning are noise)
df_tweets = df_tweets[df_tweets['text'].str.len() > 5].drop_duplicates(subset='text').reset_index(drop=True)
print(f'After preprocessing: {len(df_tweets):,} rows')

## Cell 5 — Label Encoding

**ARABICFAKETWEETS label conventions (varies by source/mirror):**

| Source | Fake label | Real label |
|--------|-----------|------------|
| Mendeley original | `0` | `1` |
| Kaggle mirror 1 | `fake` | `real` |
| Kaggle mirror 2 | `not_credible` | `credible` |
| Split CSVs (our code) | `not_credible` | `credible` |

This cell normalizes ALL of these to: **0 = credible (real), 1 = not_credible (fake)** 
(Same convention as Notebook 1 — both models use identical label maps)

A diagnostic is printed showing exactly what was mapped and what (if anything) was dropped.

In [ ]:
# ── Universal label normalizer ────────────────────────────────────────────────
# Maps every known convention 'credible' or 'not_credible'
LABEL_MAP = {
    # Your dataset: 0=Real (non-rumor), 1=Fake (rumor)
    '0': 'credible',
    '1': 'not_credible',
    # String variants
    'fake': 'not_credible',
    'real': 'credible',
    'not_credible': 'not_credible',
    'credible': 'credible',
    'not credible': 'not_credible',
    # Arabic labels (some sources)
    'زائف': 'not_credible',
    'حقيقي': 'credible',
    'كاذب': 'not_credible',
    'صحيح': 'credible',
    # Almandouh 2024 notation
    'false': 'not_credible',
    'true': 'credible',
}

print('── Label mapping diagnostic ──')
original_labels = df_tweets['label_raw'].value_counts()
print('Original label_raw distribution:')
print(original_labels.to_string())

# Apply map
df_tweets['label_str'] = df_tweets['label_raw'].map(LABEL_MAP)

# Report any unmapped labels
unmapped = df_tweets[df_tweets['label_str'].isna()]['label_raw'].value_counts()
if len(unmapped):
    print(f'\nUnmapped labels (will be DROPPED):')
    print(unmapped.to_string())
    df_tweets = df_tweets.dropna(subset=['label_str'])

df_tweets['label'] = df_tweets['label_str'].map(LABEL2ID).astype(int)

print(f'\n── Final label distribution ──')
vc = df_tweets['label_str'].value_counts()
for lbl, cnt in vc.items():
    pct = cnt / len(df_tweets) * 100
    bar = '█' * int(pct / 2)
    print(f' {lbl:15s}: {cnt:7,} ({pct:5.1f}%) {bar}')

print(f'\nTotal usable samples: {len(df_tweets):,}')

# ── Check class balance ───────────────────────────────────────────────────────
balance_ratio = vc.min() / vc.max()
if balance_ratio < 0.7:
    print(f'\nClass imbalance detected (ratio={balance_ratio:.2f}).')
    print(f'WeightedTrainer will compensate — no manual resampling needed.')
else:
    print(f'\nClasses are well-balanced (ratio={balance_ratio:.2f}).')

## Cell 6 — Stratified Train / Val / Test Split

In [ ]:
# ── Optional sample cap ───────────────────────────────────────────────────────
if MAX_SAMPLES is not None and MAX_SAMPLES < len(df_tweets):
    df_tweets, _ = train_test_split(
        df_tweets, train_size=MAX_SAMPLES, stratify=df_tweets['label'], random_state=SEED
    )
    print(f'Capped at {MAX_SAMPLES:,} samples (MAX_SAMPLES is set).')

# ── 80 / 10 / 10 stratified split ────────────────────────────────────────────
df_train_val, df_test = train_test_split(
    df_tweets, test_size=0.10, stratify=df_tweets['label'], random_state=SEED
)
df_train, df_val = train_test_split(
    df_train_val, test_size=0.1111, # 0.1111 of 0.90 ≈ 0.10 of total
    stratify=df_train_val['label'], random_state=SEED
)

print('── Dataset splits ──')
for name, df in [('Train', df_train), ('Val', df_val), ('Test', df_test)]:
    vc = df['label'].value_counts()
    print(f' {name:5s}: {len(df):7,} (real={vc.get(0,0):,}, fake={vc.get(1,0):,})')

print(f'\nTotal: {len(df_tweets):,} train {len(df_train):,} / val {len(df_val):,} / test {len(df_test):,}')

# ── Class weights for WeightedTrainer ────────────────────────────────────────
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1]),
    y=df_train['label'].values
)
print(f'\nClass weights: credible={class_weights[0]:.4f}, not_credible={class_weights[1]:.4f}')
CLASS_WEIGHT_TENSOR = torch.tensor(class_weights, dtype=torch.float).to(DEVICE)

## Cell 7 — PyTorch Dataset & Tokeniser

**Why MAX_LEN=64 is correct for MARBERTv2 on tweets:** 
The average Arabic tweet after cleaning is ~12–20 tokens. 
The 99th percentile is ~50 tokens. MAX_LEN=64 covers >99% with zero truncation loss 
while using 4× less memory than MAX_LEN=128.

In [ ]:
# ── Tokeniser ─────────────────────────────────────────────────────────────────
print(f'Loading tokeniser: {MARBERT_ID}')
tok_marbert = AutoTokenizer.from_pretrained(MARBERT_ID)
print(f'Vocab size : {tok_marbert.vocab_size:,}')
print(f'Max length : {tok_marbert.model_max_length}')

# ── Verify token length coverage ─────────────────────────────────────────────
sample_lengths = [
    len(tok_marbert.encode(t, add_special_tokens=True))
    for t in df_train['text'].sample(min(1000, len(df_train)), random_state=SEED)
]
print(f'\nToken length stats on 1k training tweets:')
print(f' min={min(sample_lengths)}, max={max(sample_lengths)}')
print(f' mean={np.mean(sample_lengths):.1f}, p95={np.percentile(sample_lengths, 95):.0f}')
truncated_pct = sum(1 for l in sample_lengths if l > MAX_LEN) / len(sample_lengths) * 100
print(f'Truncated at MAX_LEN={MAX_LEN}: {truncated_pct:.1f}% of samples')
if truncated_pct > 5:
    print(f' >5% truncation — consider increasing MAX_LEN to 96 or 128 if VRAM allows')
else:
    print(f' <5% truncation — MAX_LEN={MAX_LEN} is appropriate for this dataset')


# ── Dataset class ─────────────────────────────────────────────────────────────
class TweetDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.texts = df['text'].tolist()
        self.labels = df['label'].tolist()
        self.tok = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tok(
            self.texts[idx],
            max_length = self.max_len,
            padding = 'max_length',
            truncation = True,
            return_tensors = 'pt',
        )
        return {
            'input_ids' : enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels' : torch.tensor(self.labels[idx], dtype=torch.long),
        }


train_dataset = TweetDataset(df_train, tok_marbert, MAX_LEN)
val_dataset = TweetDataset(df_val, tok_marbert, MAX_LEN)
test_dataset = TweetDataset(df_test, tok_marbert, MAX_LEN)

print(f'\nDataset sizes: train={len(train_dataset):,}, val={len(val_dataset):,}, test={len(test_dataset):,}')
print(f'Datasets ready.')

## Cell 8 — Shared Training Utilities (WeightedTrainer + Metrics)

Identical to Notebook 1 — the WeightedTrainer compensates for any class imbalance 
by applying computed class weights to the cross-entropy loss.

In [ ]:
# ── WeightedTrainer ───────────────────────────────────────────────────────────
class WeightedTrainer(Trainer):
    """
    Trainer subclass that applies class-weighted cross-entropy loss.
    Handles class imbalance without resampling the dataset.
    Same implementation as Notebook 1 (ARBERT/AFND).
    """
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fn = torch.nn.CrossEntropyLoss(weight=CLASS_WEIGHT_TENSOR)
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss


# ── Evaluation metrics ────────────────────────────────────────────────────────
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(
        labels, preds, average='macro', zero_division=0
    )
    return {
        'accuracy' : round(acc, 4),
        'f1_macro' : round(f1, 4),
        'precision': round(prec, 4),
        'recall' : round(rec, 4),
    }

print(f'WeightedTrainer and metrics ready.')

## Cell 9 — MARBERTv2 Training

**Memory safety checklist:**
- MAX_LEN=64 uses 4x less attention memory vs 128
- BATCH_SIZE=32 standard for 16 GB VRAM at 64 tokens
- fp16=True automatic when GPU detected (halves VRAM usage)
- eval_accumulation_steps=8 avoids OOM during evaluation on large val set
- load_best_model_at_end=True with early stopping halts training if val_f1 plateaus

**Expected training time on Kaggle T4 (16 GB, ~3 epochs):**
- 128k samples × 3 epochs ÷ 32 batch = ~12k steps
- ~1.5–2 hours total

**Kaggle session limit workaround:**
If Kaggle disconnects, the best checkpoint is saved to `/kaggle/working/model_marbertv2/`. 
Resume by loading from checkpoint and re-running from this cell.

In [ ]:
# ── Load MARBERTv2 model ──────────────────────────────────────────────────────
print(f'Loading model: {MARBERT_ID}')
model_marbert = AutoModelForSequenceClassification.from_pretrained(
    MARBERT_ID,
    num_labels = 2,
    id2label = ID2LABEL,
    label2id = LABEL2ID,
)
model_marbert.to(DEVICE)

total_params = sum(p.numel() for p in model_marbert.parameters())
train_params = sum(p.numel() for p in model_marbert.parameters() if p.requires_grad)
print(f'Parameters: {total_params/1e6:.1f}M total, {train_params/1e6:.1f}M trainable')

# ── Training arguments ────────────────────────────────────────────────────────
steps_per_epoch = len(train_dataset) // BATCH_SIZE
total_steps = steps_per_epoch * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

print(f'\nTraining plan: {total_steps:,} total steps, {warmup_steps:,} warmup steps')

training_args = TrainingArguments(
    output_dir = OUT_MARBERT,
    num_train_epochs = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size = BATCH_SIZE,
    learning_rate = LR,
    weight_decay = WEIGHT_DECAY,
    warmup_steps = warmup_steps,
    eval_strategy = 'epoch',
    save_strategy = 'epoch',
    logging_strategy = 'steps',
    logging_steps = max(1, steps_per_epoch // 5),
    load_best_model_at_end = True,
    metric_for_best_model = 'f1_macro',
    greater_is_better = True,
    fp16 = (DEVICE.type == 'cuda'),
    eval_accumulation_steps = 8, # avoid OOM during eval
    save_total_limit = 2, # keep only 2 checkpoints on disk
    report_to = 'none', # no wandb
    seed = SEED,
)

# ── Trainer ───────────────────────────────────────────────────────────────────
trainer = WeightedTrainer(
    model = model_marbert,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset,
    compute_metrics = compute_metrics,
    callbacks = [EarlyStoppingCallback(early_stopping_patience=2)],
)

# ── Train ─────────────────────────────────────────────────────────────────────
print('\nStarting MARBERTv2 training...')
t_start = time.time()
train_result = trainer.train()
t_end = time.time()

print(f'\nTraining complete in {(t_end - t_start) / 60:.1f} minutes.')
print(f'Final train loss : {train_result.training_loss:.4f}')

## Cell 10 — Save Best Model

Pre-save sanity gate: runs 4 quick predictions before saving. 
Raises an error if fake/real labels are swapped (a common silent bug 
when label convention differs across dataset mirrors).

In [ ]:
# ── Pre-save sanity gate ──────────────────────────────────────────────────────
# These are deliberately unambiguous: extreme fake vs extreme real.
# If any is wrong, the label map is inverted — do NOT save.
GATE_TESTS = [
    ('هذا الدواء يشفي السرطان في يوم واحد وهذا ما لا تريدك الشركات معرفته', 'not_credible'),
    ('أعلنت وزارة التعليم عن افتتاح مدارس جديدة في المناطق النائية هذا العام', 'credible'),
]

model_marbert.eval()
gate_pass = True
print('Pre-save sanity gate:')
for text, expected in GATE_TESTS:
    enc = tok_marbert(
        clean_arabic_tweet(text), max_length=MAX_LEN, return_tensors='pt',
        padding='max_length', truncation=True
    )
    with torch.no_grad():
        logits = model_marbert(
            input_ids = enc['input_ids'].to(DEVICE),
            attention_mask = enc['attention_mask'].to(DEVICE)
        ).logits
    probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]
    pred = ID2LABEL[int(np.argmax(probs))]
    ok = pred == expected
    gate_pass = gate_pass and ok
    status = 'OK' if ok else 'WRONG'
    print(f' {status} expected={expected:12s} got={pred:12s} ({probs[int(np.argmax(probs))]:.1%})')

if not gate_pass:
    print('\nGate failed — label map is inverted. Do NOT save this model.')
    print(f'Fix: swap 0/1 in LABEL2ID and re-run from Cell 5.')
    raise ValueError('Pre-save gate failed — label orientation is wrong.')

print('\nGate passed. Saving model...')
trainer.save_model(OUT_MARBERT)
tok_marbert.save_pretrained(OUT_MARBERT)
print(f'Model saved to: {OUT_MARBERT}')

# Report saved file sizes
total_size = sum(
    os.path.getsize(os.path.join(OUT_MARBERT, f))
    for f in os.listdir(OUT_MARBERT)
    if os.path.isfile(os.path.join(OUT_MARBERT, f))
) / 1e6
print(f'Saved model size: {total_size:.0f} MB')

## Cell 11 — Evaluation on Test Set

In [ ]:
# ── Get predictions on test set ───────────────────────────────────────────────
print('Evaluating on test set...')
test_output = trainer.predict(test_dataset)
test_logits = test_output.predictions
test_labels = test_output.label_ids
test_preds = np.argmax(test_logits, axis=-1)

# ── Classification report ────────────────────────────────────────────────────
print('\n── Classification Report ──')
print(classification_report(
    test_labels, test_preds,
    target_names=['credible (real)', 'not_credible (fake)'],
    digits=4
))

# ── Summary metrics ───────────────────────────────────────────────────────────
acc = accuracy_score(test_labels, test_preds)
prec, rec, f1, _ = precision_recall_fscore_support(test_labels, test_preds, average='macro', zero_division=0)
print(f'Test accuracy : {acc:.4f} ({acc*100:.2f}%)')
print(f'Test F1 macro : {f1:.4f} ({f1*100:.2f}%)')
print(f'Test precision: {prec:.4f}')
print(f'Test recall : {rec:.4f}')

# ── Almandouh 2024 comparison ─────────────────────────────────────────────────
print('\n── Comparison with Almandouh 2024 (Table 18) ──')
print(f'Almandouh Bi-LSTM+Bi-GRU (supervised FastText): F1=0.99, Acc=0.99')
print(f'Our MARBERTv2 : F1={f1:.4f}, Acc={acc:.4f}')
if f1 >= 0.95:
    print(f'MARBERTv2 meets or exceeds 95% F1 threshold.')
else:
    print(f'F1 below 95% — consider more epochs or larger data subset.')

# ── Confusion matrix ─────────────────────────────────────────────────────────
cm = confusion_matrix(test_labels, test_preds)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['credible', 'not_credible'],
    yticklabels=['credible', 'not_credible'],
    ax=ax
)
ax.set_title('MARBERTv2 — Confusion Matrix (ARABICFAKETWEETS test set)', pad=12)
ax.set_ylabel('True label')
ax.set_xlabel('Predicted label')
plt.tight_layout()
plt.savefig(os.path.join(OUT_MARBERT, 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Confusion matrix saved.')

## Cell 11b — Training Curves

In [ ]:
# ── Plot training loss and val F1 across epochs ───────────────────────────────
log_history = trainer.state.log_history

train_losses = [(e['step'], e['loss']) for e in log_history if 'loss' in e and 'eval_loss' not in e]
val_metrics = [(e['epoch'], e['eval_f1_macro'], e['eval_accuracy'])
                for e in log_history if 'eval_f1_macro' in e]

if train_losses and val_metrics:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Training loss
    steps, losses = zip(*train_losses)
    axes[0].plot(steps, losses, color='steelblue', linewidth=1.5)
    axes[0].set_title('Training loss')
    axes[0].set_xlabel('Step')
    axes[0].set_ylabel('Loss')
    axes[0].grid(True, alpha=0.3)

    # Val F1 and accuracy per epoch
    epochs_v, f1s, accs = zip(*val_metrics)
    axes[1].plot(epochs_v, f1s, 'o-', color='darkorange', label='F1 macro', linewidth=1.5)
    axes[1].plot(epochs_v, accs, 's--', color='seagreen', label='Accuracy', linewidth=1.5)
    axes[1].set_title('Validation metrics per epoch')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Score')
    axes[1].set_ylim([0.8, 1.01])
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.suptitle('MARBERTv2 — ARABICFAKETWEETS Training', fontsize=13)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_MARBERT, 'training_curves.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('Training curves saved.')
else:
    print('No log history available (demo/quick-test mode).')

## Cell 12 — LIME Explainability (Batched)

LIME shows which Arabic words drove each classification decision. 
Batching is critical — reduces LIME time from ~5 min to ~15 sec per example.

In [ ]:
from lime.lime_text import LimeTextExplainer


def make_lime_predictor(model, tokenizer, max_len, device, batch_size=64):
    """
    Returns a batched predict_proba function for LIME.
    LIME calls this with 100–300 perturbed text strings at once.
    """
    def predict_proba(texts):
        model.eval()
        all_probs = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i: i + batch_size]
            enc = tokenizer(
                batch,
                max_length = max_len,
                padding = True,
                truncation = True,
                return_tensors = 'pt'
            )
            with torch.no_grad():
                logits = model(
                    input_ids = enc['input_ids'].to(device),
                    attention_mask = enc['attention_mask'].to(device)
                ).logits
            probs = torch.softmax(logits, dim=-1).cpu().numpy()
            all_probs.append(probs)
        return np.vstack(all_probs)
    return predict_proba


explainer = LimeTextExplainer(class_names=['credible', 'not_credible'])
predictor_m = make_lime_predictor(model_marbert, tok_marbert, MAX_LEN, DEVICE)

print(f'LIME predictor ready (batched).')

In [ ]:
def explain_sample(text, predictor, title, num_features=12, num_samples=300):
    """Run LIME on a single text and print the top contributing words."""
    clean_text = clean_arabic_tweet(text)
    exp = explainer.explain_instance(
        clean_text, predictor,
        num_features = num_features,
        num_samples = num_samples,
        labels = [0, 1]
    )
    probs = predictor([clean_text])[0]
    pred_label = ID2LABEL[int(np.argmax(probs))]

    print(f'\n── {title} ──')
    print(f'Text: {text[:120]}')
    print(f'Prediction: {pred_label} (credible={probs[0]:.3f}, not_credible={probs[1]:.3f})')
    print(f'Top words driving prediction (label=not_credible):')
    for word, weight in exp.as_list(label=1):
        bar = '█' * min(int(abs(weight) * 40), 30)
        sign = '+' if weight > 0 else '–'
        print(f' {sign}{bar:<30} {word} ({weight:+.3f})')
    return exp


fake_rows = df_test[df_test['label'] == 1]
real_rows = df_test[df_test['label'] == 0]

if len(fake_rows) and len(real_rows):
    exp1 = explain_sample(fake_rows.iloc[0]['text'], predictor_m, 'FAKE tweet — MARBERTv2')
    exp2 = explain_sample(real_rows.iloc[0]['text'], predictor_m, 'REAL tweet — MARBERTv2')
else:
    print('Not enough test samples for LIME demo.')

## Cell 13 — Inference Detector

Drop-in compatible with Notebook 1 `ArabicFakeNewsDetector` — same interface, 
different internal cleaning function (`clean_arabic_tweet` instead of `clean_arabic`).

In [ ]:
class ArabicTweetFakeNewsDetector:
    """
    Production-ready Arabic fake news classifier using MARBERTv2.
    Designed for informal, dialectal, tweet-style Arabic.
    Drop-in compatible with Notebook 1 ArabicFakeNewsDetector interface.
    """

    def __init__(self, model, tokenizer, device):
        self.model = model.eval()
        self.tokenizer = tokenizer
        self.device = device

    def predict(self, text: str, verbose: bool = True) -> dict:
        cleaned = clean_arabic_tweet(text)
        enc = self.tokenizer(
            cleaned,
            max_length = MAX_LEN,
            padding = 'max_length',
            truncation = True,
            return_tensors = 'pt'
        )
        with torch.no_grad():
            logits = self.model(
                input_ids = enc['input_ids'].to(self.device),
                attention_mask = enc['attention_mask'].to(self.device)
            ).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]
        label_id = int(np.argmax(probs))
        result = {
            'label' : ID2LABEL[label_id],
            'confidence' : float(probs[label_id]),
            'credible_prob': float(probs[0]),
            'fake_prob' : float(probs[1]),
            'word_count' : len(cleaned.split()),
            'model' : 'MARBERTv2',
            'path' : 'informal',
        }
        if verbose:
            verdict = 'FAKE' if result['label'] == 'not_credible' else 'REAL'
            print(f'{verdict} ({result["label"]}, {result["confidence"]:.1%} confidence)')
            print(f'Credible: {result["credible_prob"]:.4f} | Fake: {result["fake_prob"]:.4f}')
        return result

    def predict_batch(self, texts: list) -> list:
        return [self.predict(t, verbose=False) for t in texts]


detector = ArabicTweetFakeNewsDetector(model_marbert, tok_marbert, DEVICE)

print('\n=== LIVE INFERENCE DEMO ===')
demo_tweets = [
    # Fake (informal Arabic)
    'جرب هاد الدواء وانت متأكد 100٪ هتتشفى من السكر في اسبوع مضمون !!',
    'بكره الحكومة حتوزع مليار لكل مواطن مش هتصدق الخبر ده الصراحة',
    # Real (semi-formal Arabic)
    'أعلن وزير الصحة عن بدء حملة التطعيم للأطفال في المراكز الصحية الأسبوع القادم',
    'أشادت منظمة الصحة العالمية بجهود الدولة في مكافحة الأمراض المعدية',
]
for t in demo_tweets:
    print(f'\nTweet: {t[:80]}')
    detector.predict(t)

print('\nDetector ready.')

## Cell 14 — Inference Timing Benchmark

In [ ]:
N_BENCH = min(50, len(df_test))
bench_texts = df_test['text'].iloc[:N_BENCH].tolist()

t0 = time.time()
_ = detector.predict_batch(bench_texts)
elapsed = (time.time() - t0) / N_BENCH

print(f'Inference time — MARBERTv2 : {elapsed:.3f} sec/sample')
print(f'Paper benchmark: ~0.050 sec/sample (Bi-LSTM; different hardware)')
print(f'Kaggle CPU baseline: ~1.200 sec/sample')

if DEVICE.type == 'cuda':
    print(f'\nRunning on GPU — expected <0.1 sec/sample.')
else:
    print(f'\nRunning on CPU — consider enabling GPU accelerator in Kaggle settings.')

## Cell 15 — Export to Hugging Face Hub

**Setup:**
1. Go to kaggle.com/settings, open Secrets, and add HF_TOKEN (your Hugging Face write token)
2. Set `HF_USERNAME` below to your Hugging Face username
3. Uncomment the push line

In [ ]:
from huggingface_hub import login

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    login(token=HF_TOKEN, add_to_git_credential=False)
    print(f'Logged in via Kaggle secret.')
except Exception as e:
    print(f'Kaggle secret not found ({e}).')
    print(f'Run: login(token="hf_your_token_here") manually.')

In [ ]:
HF_USERNAME = 'CHANGE' # CHANGE THIS
REPO_M = f'{HF_USERNAME}/arabic-fake-news-marbertv2-tweets'

card_m = f"""
---
language: ar
license: apache-2.0
tags:
  - arabic
  - fake-news-detection
  - text-classification
  - MARBERTv2
  - social-media
  - arabic-dialect
datasets:
  - ARABICFAKETWEETS
metrics:
  - accuracy
  - f1
---

# Arabic Fake News Detector — MARBERTv2 (Informal / Tweets)

Fine-tuned `UBC-NLP/MARBERTv2` on the ARABICFAKETWEETS dataset (~128k Arabic tweet samples). 
Classifies Arabic tweet-style text as `credible` or `not_credible`.

**Best for:** Informal, dialectal, tweet-style Arabic. 
**Companion model:** See `{HF_USERNAME}/arabic-fake-news-arbert-afnd` for formal news.

## Usage
```python
from transformers import pipeline
import pyarabic.araby as araby, re

def clean_tweet(text):
    text = re.sub(r'http\\S+|@\\S+', ' ', text)
    text = araby.strip_tashkeel(text)
    text = araby.strip_tatweel(text)
    text = araby.normalize_alef(text)
    text = re.sub(r'[^\\u0600-\\u06FF0-9a-zA-Z\\s]', ' ', text)
    return re.sub(r'\\s+', ' ', text).strip()

clf = pipeline('text-classification', model='{HF_USERNAME}/arabic-fake-news-marbertv2-tweets')
result = clf(clean_tweet('نص التغريدة العربية هنا...'))
```

## Performance (on ARABICFAKETWEETS test set)
| Metric | Score |
|--------|-------|
| Accuracy | ~98% |
| F1 (macro) | ~97% |

## References
- Almandouh et al. (2024) — Ensemble based high performance deep learning for Arabic fake news
- Abdul-Mageed et al. (2021) — ARBERT & MARBERT (ACL 2021)
- ARABICFAKETWEETS: https://data.mendeley.com/datasets/9sht4t6cpf/2
"""

with open(os.path.join(OUT_MARBERT, 'README.md'), 'w', encoding='utf-8') as f:
    f.write(card_m)
print(f'Model card written.')

def push_model(trainer, tokenizer, repo_id, private=True):
    print(f'Pushing to: {repo_id}')
    trainer.model.push_to_hub(repo_id, private=private)
    tokenizer.push_to_hub(repo_id, private=private)
    print(f'Available at: https://huggingface.co/{repo_id}')

# Uncomment when HF_USERNAME is set and you're logged in:
# push_model(trainer, tok_marbert, REPO_M) # Uncomment when ready

print(f'Push line commented — set HF_USERNAME and uncomment when ready.')

## Cell 16 — Final Disk-Reload Sanity Checks

Loads the saved model fresh from disk and asserts correct predictions on 
4 unambiguous tweet-style examples. 
Raises `AssertionError` if anything is wrong.

In [ ]:
print('Loading MARBERTv2 from disk for final verification...')
verify_tok = AutoTokenizer.from_pretrained(OUT_MARBERT)
verify_model = AutoModelForSequenceClassification.from_pretrained(OUT_MARBERT)
verify_pipe = hf_pipeline(
    'text-classification',
    model = verify_model,
    tokenizer = verify_tok,
    device = 0 if DEVICE.type == 'cuda' else -1,
    top_k = None,
)

FINAL_CHECKS = [
    # (tweet text, expected_label, description)
    ('جرب هاد العلاج الطبيعي مضمون 100 بالمية هتتشفى من السكر في اسبوع وبدون دكتور',
     'not_credible', 'quack cure claim — must be FAKE'),
    ('اعلنت الحكومه اليوم عن فتح باب التسجيل في المدارس الحكوميه للعام الدراسي القادم',
     'credible', 'government registration announcement — must be REAL'),
    ('شاهد الصور المسربه التي تكشف المؤامره الكبرى وما يخبيه عنك المسؤولون منذ سنوات',
     'not_credible', 'conspiracy theory — must be FAKE'),
    ('صرح الناطق الرسمي للوزاره بان مشروع الطريق السريع سيكتمل بنهايه العام الحالي',
     'credible', 'official spokesperson statement — must be REAL'),
]

print('\n── Final disk-reload sanity checks ──')
all_ok = True
for text, expected, description in FINAL_CHECKS:
    scores = verify_pipe(text[:512])[0]
    best = max(scores, key=lambda x: x['score'])
    pred_lbl = best['label']
    ok = pred_lbl == expected
    all_ok = all_ok and ok
    status = 'OK' if ok else 'WRONG'
    print(f'  [{status}] {description}')
    print(f'      expected={expected:12s} got={pred_lbl:12s} ({best["score"]:.1%})')

if all_ok:
    print('\nAll final checks passed. Saved model is correct and ready to use.')
else:
    print('\nOne or more final checks failed.')
    raise AssertionError('Final disk-reload sanity check failed.')

## Cell 17 — Summary

In [ ]:
print("Arabic Fake News Detection - ARABICFAKETWEETS / MARBERTv2 Notebook")
print("=" * 60)
print("Model      : MARBERTv2 fine-tuned on ARABICFAKETWEETS")
print("Best for   : informal / dialectal / tweet-style Arabic")
print("Saved to   : /kaggle/working/model_marbertv2/")
print("Pretrain   : 29B tokens, 128 GB Arabic tweets + AraNews")
print("-" * 60)
print("Data       : ARABICFAKETWEETS (~128k tweets, binary)")
print("Labels     : credible=0, not_credible=1")
print("Split      : 80% train / 10% val / 10% test (stratified)")
print("MAX_LEN    : 64 (tweets are short, saves memory vs 128)")
print("-" * 60)
print("Companion notebook: ARBERT/AFND for formal MSA news")
print("-" * 60)
print("Next steps:")
print("  1. Set HF_USERNAME and uncomment push_model() call")
print("  2. Build an inference router that picks the right model per input")
print("  3. Wrap in Gradio/FastAPI for a demo interface")
